# Nested five-fold selection of post-training blend and ε on O-HAZE

Inference-only evaluation using all 45 O-HAZE pairs and fixed Stage 4 checkpoints for seeds 17, 43, and 73. Inputs use 512×512 radiometric preprocessing and the observed-hazy ASM equation. Candidate grid: six blend weights `[0, .02, .05, .08, .10, .20]` × five ε values `[.01, .05, .10, .20, .50]`.

For each outer fold, choose one shared `(blend, ε)` pair on the other 36 images by mean PSNR across all three checkpoints; mean SSIM breaks exact PSNR ties. Score the chosen pair once on the nine outer images and compare with fixed `(.08, .10)`. Five deterministic image-disjoint folds cover all 45 pairs. The bootstrap resamples images and keeps each image's three seed outcomes together.

**Scope:** this selects an inference-time output mix over fixed checkpoints. It does not select or validate a training blend, retrain models, or estimate a component's causal benefit. The manuscript already reports O-HAZE at ε=.10; this is not pristine external confirmation if prior O-HAZE results influenced parameter choices. Bootstrap intervals condition on the realized fold assignments and selections; they do not repeat the selection procedure inside each bootstrap draw.

Kaggle paths follow the reference `sweep-analysis.ipynb` and prior O-HAZE evaluation. Run cells in order. No inference runs in this notebook's creation or validation.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys
import time

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

PACKAGE = Path("/kaggle/input/datasets/randomguy28/reviewer-test-packge/reviewer_test_package")
OHAZE_ROOT = Path("/kaggle/input/datasets/philiphofmann/o-haze/O-HAZY")
HAZY_DIR = OHAZE_ROOT / "hazy"
GT_DIR = OHAZE_ROOT / "GT"
OUTPUT_ROOT = Path("/kaggle/working/nested_ohaze_blend_epsilon_cv")
SCRIPT_ROOT = PACKAGE / "code"
CHECKPOINTS = {
    seed: PACKAGE / f"models/full/seed{seed}/stage4_best.keras"
    for seed in (17, 43, 73)
}
BLEND_WEIGHTS = (0.00, 0.02, 0.05, 0.08, 0.10, 0.20)
EPSILONS = (0.01, 0.05, 0.10, 0.20, 0.50)
BASELINE = (0.08, 0.10)
IMAGE_SIZE = 512
PREPROCESSING = "radiometric"
N_FOLDS = 5
FOLD_SEED = 42
BOOTSTRAP_SAMPLES = 10_000
BOOTSTRAP_SEED = 20261014

assert PACKAGE.is_dir(), f"Check Kaggle package path: {PACKAGE}"
assert HAZY_DIR.is_dir(), f"Check O-HAZE hazy directory: {HAZY_DIR}"
assert GT_DIR.is_dir(), f"Check O-HAZE GT directory: {GT_DIR}"
assert all(path.is_file() for path in CHECKPOINTS.values()), "Missing fixed Stage 4 checkpoint."

sys.path.insert(0, str(SCRIPT_ROOT))
from eca_ldnet.data import get_gt_path, load_image_pair
from eca_ldnet.layers import CUSTOM_OBJECTS

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

image_exts = {".png", ".jpg", ".jpeg", ".bmp"}
pair_records, unmatched = [], []
for hazy_path in sorted(p for p in HAZY_DIR.iterdir() if p.suffix.lower() in image_exts):
    gt_path = get_gt_path(hazy_path.name, str(GT_DIR))
    if gt_path is None:
        unmatched.append(hazy_path.name)
    else:
        pair_records.append({
            "image": hazy_path.name,
            "hazy_path": str(hazy_path),
            "clean_path": str(gt_path),
        })
missing = [row[key] for row in pair_records for key in ("hazy_path", "clean_path") if not Path(row[key]).is_file()]
print("Matched pairs:", len(pair_records), "| unmatched:", unmatched, "| missing:", missing[:5])
assert len(pair_records) == 45, f"Expected all 45 O-HAZE pairs; found {len(pair_records)}."
assert not unmatched and not missing, "Resolve all pair/path issues before inference."
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## One forward pass per image and checkpoint

Cache raw prediction, atmospheric light, and transmission once. Recombine cached outputs across the 30 fixed grid points; do not retrain. This cell writes one metric row per image, seed, blend, and ε.

In [ ]:
all_rows = []
checkpoint_hashes = {seed: sha256_file(path) for seed, path in CHECKPOINTS.items()}
started = time.perf_counter()

for seed, checkpoint in CHECKPOINTS.items():
    print(f"Loading fixed seed {seed}: {checkpoint.name}")
    model = tf.keras.models.load_model(str(checkpoint), custom_objects=CUSTOM_OBJECTS, compile=False)
    if model.input_shape[1] is not None:
        assert model.input_shape[1] == IMAGE_SIZE, f"Checkpoint input height is {model.input_shape[1]}, expected {IMAGE_SIZE}."
    heads = tf.keras.Model(
        model.input,
        [model.get_layer("raw_sigmoid").output,
         model.get_layer("atm_light").output,
         model.get_layer("t_full").output],
    )

    for index, pair in enumerate(pair_records, start=1):
        hazy, clean = load_image_pair(
            pair["hazy_path"], pair["clean_path"], size=IMAGE_SIZE,
            preprocessing=PREPROCESSING,
        )
        if hazy is None or clean is None:
            raise RuntimeError(f"Could not read pair: {pair['image']}")
        raw_b, atm_b, transmission_b = heads.predict(hazy[None, ...], verbose=0)
        raw = raw_b[0]
        atm = atm_b[0].reshape(1, 1, 3)
        transmission = transmission_b[0]

        for epsilon in EPSILONS:
            # Manuscript-aligned observed-hazy ASM estimate.
            asm_estimate = np.clip((hazy - atm) / (transmission + epsilon) + atm, 0.0, 1.0)
            for blend in BLEND_WEIGHTS:
                prediction = np.clip(raw * (1.0 - blend) + asm_estimate * blend, 0.0, 1.0).astype(np.float32)
                psnr = float(tf.image.psnr(prediction[None], clean[None], 1.0).numpy().reshape(-1)[0])
                ssim = float(tf.image.ssim(prediction[None], clean[None], 1.0).numpy().reshape(-1)[0])
                all_rows.append({
                    "seed": seed, "image": pair["image"], "blend": blend,
                    "epsilon": epsilon, "psnr": psnr, "ssim": ssim,
                    "source": "observed_hazy",
                })
        if index % 5 == 0 or index == len(pair_records):
            print(f"  seed {seed}: {index}/{len(pair_records)} pairs")

    del heads, model
    tf.keras.backend.clear_session()

metrics = pd.DataFrame(all_rows)
expected = len(pair_records) * len(CHECKPOINTS) * len(BLEND_WEIGHTS) * len(EPSILONS)
assert len(metrics) == expected, f"Expected {expected} rows; got {len(metrics)}."
assert metrics.groupby(["seed", "image", "blend", "epsilon"]).size().eq(1).all()
metrics_path = OUTPUT_ROOT / "all_seeds_per_image_grid.csv"
metrics.to_csv(metrics_path, index=False)
print(f"Saved {len(metrics):,} rows: {metrics_path}")
print(f"Inference and scoring elapsed: {(time.perf_counter() - started) / 60:.1f} min")

## Nested selection and held-out scoring

Folds are assigned by image filename with a fixed seed. The single shared grid point per outer fold is chosen from the 36 inner images, averaging across the three fixed checkpoints. Exact PSNR ties use SSIM; any remaining ties resolve to lower blend then lower ε for deterministic output.

In [ ]:
images = np.array(sorted(metrics["image"].unique()))
assert len(images) == 45
fold_rng = np.random.default_rng(FOLD_SEED)
shuffled = images.copy()
fold_rng.shuffle(shuffled)
fold_groups = np.array_split(shuffled, N_FOLDS)
image_to_fold = {
    image: fold
    for fold, group in enumerate(fold_groups, start=1)
    for image in group
}
metrics["outer_fold"] = metrics["image"].map(image_to_fold)
fold_assignments = pd.DataFrame([
    {"image": image, "outer_fold": image_to_fold[image]}
    for image in images
])
fold_assignments.to_csv(OUTPUT_ROOT / "fold_assignments.csv", index=False)
assert fold_assignments.groupby("outer_fold")["image"].nunique().eq(9).all()

fold_rows, oof_rows = [], []
for fold in range(1, N_FOLDS + 1):
    inner = metrics.loc[metrics["outer_fold"] != fold]
    outer = metrics.loc[metrics["outer_fold"] == fold]
    inner_scores = (
        inner.groupby(["blend", "epsilon"], as_index=False)
        .agg(inner_psnr=("psnr", "mean"), inner_ssim=("ssim", "mean"))
        .sort_values(
            ["inner_psnr", "inner_ssim", "blend", "epsilon"],
            ascending=[False, False, True, True],
            kind="mergesort",
        )
    )
    best = inner_scores.iloc[0]
    selected_blend, selected_epsilon = float(best["blend"]), float(best["epsilon"])

    selected = outer.loc[
        (outer["blend"] == selected_blend) & (outer["epsilon"] == selected_epsilon),
        ["image", "seed", "psnr", "ssim"],
    ].rename(columns={"psnr":"selected_psnr", "ssim":"selected_ssim"})
    baseline = outer.loc[
        (outer["blend"] == BASELINE[0]) & (outer["epsilon"] == BASELINE[1]),
        ["image", "seed", "psnr", "ssim"],
    ].rename(columns={"psnr":"baseline_psnr", "ssim":"baseline_ssim"})
    fold_oof = selected.merge(baseline, on=["image", "seed"], validate="one_to_one")
    fold_oof["outer_fold"] = fold
    fold_oof["selected_blend"] = selected_blend
    fold_oof["selected_epsilon"] = selected_epsilon
    fold_oof["delta_psnr"] = fold_oof["selected_psnr"] - fold_oof["baseline_psnr"]
    fold_oof["delta_ssim"] = fold_oof["selected_ssim"] - fold_oof["baseline_ssim"]
    oof_rows.extend(fold_oof.to_dict("records"))

    fold_rows.append({
        "outer_fold": fold,
        "n_inner_images": inner["image"].nunique(),
        "n_outer_images": outer["image"].nunique(),
        "selected_blend": selected_blend,
        "selected_epsilon": selected_epsilon,
        "inner_psnr_at_selected": float(best["inner_psnr"]),
        "inner_ssim_at_selected": float(best["inner_ssim"]),
        "outer_selected_psnr": float(fold_oof["selected_psnr"].mean()),
        "outer_baseline_psnr": float(fold_oof["baseline_psnr"].mean()),
        "outer_delta_psnr": float(fold_oof["delta_psnr"].mean()),
        "outer_selected_ssim": float(fold_oof["selected_ssim"].mean()),
        "outer_baseline_ssim": float(fold_oof["baseline_ssim"].mean()),
        "outer_delta_ssim": float(fold_oof["delta_ssim"].mean()),
    })

fold_results = pd.DataFrame(fold_rows)
oof = pd.DataFrame(oof_rows)
assert len(oof) == 45 * len(CHECKPOINTS)
fold_results.to_csv(OUTPUT_ROOT / "fold_selection_and_scores.csv", index=False)
oof.to_csv(OUTPUT_ROOT / "out_of_fold_per_image.csv", index=False)
display(fold_results.round(5))

## Paired image-cluster bootstrap and outputs

Average the three seed outcomes within each O-HAZE image, then resample 45 image clusters. This preserves pairing between nested selection and fixed `(.08, .10)` baseline. Intervals describe image variability conditional on these folds and fitted selections.

In [ ]:
image_means = (
    oof.groupby("image", as_index=False)
    .agg(
        selected_psnr=("selected_psnr", "mean"), baseline_psnr=("baseline_psnr", "mean"),
        selected_ssim=("selected_ssim", "mean"), baseline_ssim=("baseline_ssim", "mean"),
    )
)
for metric in ("psnr", "ssim"):
    image_means[f"delta_{metric}"] = image_means[f"selected_{metric}"] - image_means[f"baseline_{metric}"]

boot_rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_summary = {}
for metric in ("psnr", "ssim"):
    values = image_means[f"delta_{metric}"].to_numpy()
    indices = boot_rng.integers(0, len(values), size=(BOOTSTRAP_SAMPLES, len(values)))
    boot_means = values[indices].mean(axis=1)
    bootstrap_summary[metric] = {
        "mean_delta": float(values.mean()),
        "ci95_low": float(np.quantile(boot_means, 0.025)),
        "ci95_high": float(np.quantile(boot_means, 0.975)),
        "n_image_clusters": int(len(values)),
    }
bootstrap_frame = pd.DataFrame([{"metric": metric, **stats} for metric, stats in bootstrap_summary.items()])
bootstrap_frame.to_csv(OUTPUT_ROOT / "paired_image_cluster_bootstrap.csv", index=False)
display(bootstrap_frame)
print("Selected pairs by fold:")
display(fold_results[["outer_fold", "selected_blend", "selected_epsilon"]])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
axes[0].scatter(
    fold_results["selected_blend"], fold_results["selected_epsilon"],
    c=fold_results["outer_fold"], cmap="viridis", s=85, edgecolor="black",
)
for row in fold_results.itertuples():
    axes[0].annotate(f"F{row.outer_fold}", (row.selected_blend, row.selected_epsilon), xytext=(5, 5), textcoords="offset points")
axes[0].scatter([BASELINE[0]], [BASELINE[1]], marker="x", s=100, color="red", label="Fixed (.08, .10)")
axes[0].set(xlabel="Selected blend", ylabel="Selected ε", title="Inner-fold selections")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

labels = ["PSNR (dB)", "SSIM"]
means = [bootstrap_summary[k]["mean_delta"] for k in ("psnr", "ssim")]
lows = [bootstrap_summary[k]["ci95_low"] for k in ("psnr", "ssim")]
highs = [bootstrap_summary[k]["ci95_high"] for k in ("psnr", "ssim")]
axes[1].errorbar(
    labels, means,
    yerr=[[m-lo for m,lo in zip(means,lows)], [hi-m for m,hi in zip(means,highs)]],
    fmt="o", capsize=5, color="#C0504D",
)
axes[1].axhline(0, color="black", linewidth=1, linestyle="--")
axes[1].set(ylabel="Nested selection minus fixed (.08, .10)", title="Paired image-cluster bootstrap 95% CI")
axes[1].grid(axis="y", alpha=0.25)
fig.tight_layout()
plot_path = OUTPUT_ROOT / "nested_blend_epsilon_summary.png"
fig.savefig(plot_path, dpi=180, bbox_inches="tight")
plt.show()

run_manifest = {
    "dataset": "O-HAZE",
    "n_pairs": len(pair_records),
    "image_size": IMAGE_SIZE,
    "preprocessing": PREPROCESSING,
    "asm_source": "observed_hazy",
    "blend_grid": list(BLEND_WEIGHTS),
    "epsilon_grid": list(EPSILONS),
    "fixed_baseline": {"blend": BASELINE[0], "epsilon": BASELINE[1]},
    "seeds": list(CHECKPOINTS),
    "checkpoint_sha256": checkpoint_hashes,
    "outer_folds": N_FOLDS,
    "fold_assignment_seed": FOLD_SEED,
    "selection_rule": "One shared blend-epsilon pair per outer fold; maximize mean PSNR over inner images and fixed seeds; mean SSIM exact-tie-break; then lower blend and epsilon for determinism.",
    "bootstrap_unit": "image; three seed outcomes retained together",
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "bootstrap_scope": "Conditional on realized folds and selections; selection is not repeated inside bootstrap draws.",
    "training_performed": False,
    "interpretation": "Inference-time output-mix selection over fixed checkpoints only; does not validate a training blend or causal component advantage.",
    "prior_result_caveat": "O-HAZE at epsilon=0.10 was previously reported; not pristine external confirmation if prior O-HAZE outcomes influenced parameter choices.",
    "paired_bootstrap": bootstrap_summary,
    "outputs": {
        "all_grid_metrics": str(metrics_path),
        "fold_assignments": str(OUTPUT_ROOT / "fold_assignments.csv"),
        "fold_selection_and_scores": str(OUTPUT_ROOT / "fold_selection_and_scores.csv"),
        "out_of_fold_per_image": str(OUTPUT_ROOT / "out_of_fold_per_image.csv"),
        "paired_bootstrap": str(OUTPUT_ROOT / "paired_image_cluster_bootstrap.csv"),
        "plot": str(plot_path),
    },
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
}
manifest_path = OUTPUT_ROOT / "run_manifest.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2))
print("Saved plot:", plot_path)
print("Saved manifest:", manifest_path)

## Package outputs

Create a ZIP of all files in `/kaggle/working/nested_ohaze_blend_epsilon_cv/`. The archive is saved beside that folder.

In [ ]:
import shutil

archive_path = shutil.make_archive(
    str(OUTPUT_ROOT), "zip", root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name
)
assert Path(archive_path).is_file(), f"ZIP was not created: {archive_path}"
print(f"Created ZIP: {archive_path}")